# Random Muon Scan Locations

This notebook plots the 1000 known random source positions generated for `macros/random_muon_scan.mac`. The CSV is the source of truth because it was written by the same generator that created the macro.

In [ ]:
from pathlib import Path
import re

import matplotlib.pyplot as plt
import pandas as pd

POSITIONS_CSV = Path("macros/random_muon_scan_positions.csv")
RANDOM_MAC = Path("macros/random_muon_scan.mac")
OUTPUT_PNG = Path("analysis/random_muon_scan_locations.png")

## Load Positions

In [ ]:
positions = pd.read_csv(POSITIONS_CSV)
positions.head()

In [ ]:
print(f"positions in CSV: {len(positions)}")
print(positions[["true_x_cm", "true_z_cm"]].describe())

## Cross-Check Against Macro

In [ ]:
position_re = re.compile(
    r"^/gps/position\s+"
    r"(?P<x>[-+0-9.eE]+)\s+"
    r"(?P<y>[-+0-9.eE]+)\s+"
    r"(?P<z>[-+0-9.eE]+)\s+"
    r"(?P<unit>\S+)"
)

macro_positions = []
for line in RANDOM_MAC.read_text().splitlines():
    match = position_re.match(line.strip())
    if match:
        macro_positions.append(match.groupdict())

print(f"/gps/position lines in macro: {len(macro_positions)}")
print(f"/run/beamOn 1 lines in macro: {sum(line.strip() == '/run/beamOn 1' for line in RANDOM_MAC.read_text().splitlines())}")

## Plot Locations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
sc = ax.scatter(
    positions["true_x_cm"],
    positions["true_z_cm"],
    c=positions["event_id"],
    s=14,
    cmap="viridis",
    alpha=0.85,
    linewidths=0,
)
ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-50, 50)
ax.set_ylim(-50, 50)
ax.set_xlabel("x [cm]")
ax.set_ylabel("z [cm]")
ax.set_title("Random single-muon source positions")
ax.grid(True, alpha=0.25)
cbar = fig.colorbar(sc, ax=ax)
cbar.set_label("event id")
fig.tight_layout()
fig.savefig(OUTPUT_PNG, dpi=180)
OUTPUT_PNG